In [9]:
!g++ --version

g++ (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



In [10]:
!nvidia-smi;
!nvcc --version;

Fri Sep  4 15:46:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
%pip install -q meson ninja networkit scipy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 70.5 MB/s eta 0:00:00:00:01:01


In [13]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
test -d "$PROJECT/.git"
git -C "$PROJECT" pull --ff-only


Already up to date.


## Test semi-completo CPU/GPU

Le celle seguenti compilano il progetto, generano un piccolo grafo sintetico ed eseguono lo stesso workload con i backend sequenziale, OpenMP e CUDA. Gli output numerici vengono confrontati automaticamente con una tolleranza per i calcoli floating point.

In [14]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
cd "$PROJECT"

echo '=== GPU disponibile ==='
nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader
echo '=== Toolchain CUDA ==='
nvcc --version

echo '=== Configurazione e compilazione ==='
if [ -d builddir/meson-private ]; then
  meson setup --reconfigure builddir
else
  meson setup builddir
fi
meson compile -C builddir
./builddir/gnn --help


=== GPU disponibile ===
Tesla T4, 580.82.07, 15360 MiB
=== Toolchain CUDA ===
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
=== Configurazione e compilazione ===
Regenerating configuration from scratch: Build directory has been generated with Meson version 1.11.1, which is incompatible with the current version 1.12.0.
The Meson build system
Version: 1.12.0
Source dir: /content/drive/MyDrive/cuda-gnn-inference
Build dir: /content/drive/MyDrive/cuda-gnn-inference/builddir
Build type: native build
Project name: cuda-gnn-inference
Project version: 0.1
C++ compiler for the host machine: c++ (gcc 11.4.0 "c++ (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0")
C++ linker for the host machine: c++ ld.bfd 2.38
Host machine cpu family: x86_64
Host machine cpu: x86_64
Run-time dependency OpenMP found: YES 4.5
Message: OpenMP found. Enabling the pa

In [15]:
%%bash
set -euo pipefail
PROJECT=/content/drive/MyDrive/cuda-gnn-inference
TEST_DATA=/content/cuda-gnn-semi-test
mkdir -p "$TEST_DATA"
cd "$PROJECT"

echo '=== Generazione workload sintetico ==='
python scripts/synthetic_generator.py \
  --type barabasi_albert \
  --nodes 256 \
  --feature_dim 32 \
  --m 4 \
  --seed 42 \
  --out_prefix "$TEST_DATA/graph"
ls -lh "$TEST_DATA/graph.bin_graph" "$TEST_DATA/graph_feats.bin_matrix"


=== Generazione workload sintetico ===
[+] Graph saved successfully in '/content/cuda-gnn-semi-test/graph.bin_graph' (256 nodes, 2024 edges).
[+] Matrix 256x32 saved successfully in '/content/cuda-gnn-semi-test/graph_feats.bin_matrix'.
-rw-r--r-- 1 root root 18K Sep  4 15:47 /content/cuda-gnn-semi-test/graph.bin_graph
-rw-r--r-- 1 root root 33K Sep  4 15:47 /content/cuda-gnn-semi-test/graph_feats.bin_matrix


In [17]:
from pathlib import Path
import re
import subprocess
import numpy as np

project = Path('/content/drive/MyDrive/cuda-gnn-inference')
executable = project / 'builddir' / 'gnn'
test_data = Path('/content/cuda-gnn-semi-test')

def run_backend(mode, arguments=()):
    command = [str(executable), mode, *map(str, arguments)]
    result = subprocess.run(command, cwd=project, text=True, capture_output=True)
    print(f'\n$ {" ".join(command)}')
    print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='')
    if result.returncode != 0:
        raise RuntimeError(f'{mode} terminato con codice {result.returncode}')
    rows = []
    for line in result.stdout.splitlines():
        match = re.fullmatch(r'\s*\[([^]]+)\]', line)
        if match:
            rows.append([float(value) for value in match.group(1).split(',')])
    if not rows:
        raise RuntimeError(f'{mode} non ha prodotto righe numeriche')
    return np.asarray(rows, dtype=np.float32)

def compare_case(name, arguments=()):
    print(f'\n========== {name} ==========')
    sequential = run_backend('sequential', arguments)
    parallel = run_backend('parallel', arguments)
    cuda = run_backend('cuda', arguments)
    np.testing.assert_allclose(parallel, sequential, rtol=1e-5, atol=1e-5)
    np.testing.assert_allclose(cuda, sequential, rtol=1e-5, atol=1e-5)
    print(f'OK: {name} - righe stampate equivalenti, shape confrontata {cuda.shape}')

compare_case('Demo integrato')
compare_case(
    'Grafo sintetico caricato da file',
    (test_data / 'graph.bin_graph', test_data / 'graph_feats.bin_matrix'),
)
print('\nTUTTI I TEST SONO PASSATI')



========== Demo integrato ==========

$ /content/drive/MyDrive/cuda-gnn-inference/builddir/gnn sequential
Sequential GCN -> GraphSAGE -> GCN output (nodes=2, input_features=3, output_features=2):
  [10, 11]
  [25, 26]

$ /content/drive/MyDrive/cuda-gnn-inference/builddir/gnn parallel
Parallel GCN -> GraphSAGE -> GCN output (nodes=2, input_features=3, output_features=2, up to 2 OpenMP threads):
  [10, 11]
  [25, 26]

$ /content/drive/MyDrive/cuda-gnn-inference/builddir/gnn cuda
CUDA GCN -> GraphSAGE -> GCN output (nodes=2, input_features=3, output_features=2):
  [10, 11]
  [25, 26]
OK: Demo integrato - righe stampate equivalenti, shape confrontata (2, 2)

========== Grafo sintetico caricato da file ==========

$ /content/drive/MyDrive/cuda-gnn-inference/builddir/gnn sequential /content/cuda-gnn-semi-test/graph.bin_graph /content/cuda-gnn-semi-test/graph_feats.bin_matrix
Loaded-data sequential output (nodes=256, input_features=32, output_features=1):
  [0.0699893]
  [0.0127716]
  [-0.15